# Aerial Building Segmentation — Results & Visualisation

This notebook walks through the full experiment pipeline:
1. Mount Google Drive and install dependencies
2. Download the INRIA dataset automatically via Kaggle
3. Explore the dataset
4. Run training for all three architectures
5. Load results and display a comparison table
6. Visualise qualitative predictions side-by-side

**Training scripts live in `src/` and are run with `!python`.  
This notebook is for exploration, monitoring, and visualisation.**

## 1. Setup

Mount Google Drive (checkpoints will be saved here after every epoch) and clone the project repo.  Drive mounting prompts for authorisation on the first run — follow the link and paste the code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/aerial-segmentation'
CKPT_BASE  = f'{DRIVE_BASE}/checkpoints'
REPO_DIR   = '/content/aerial-segmentation'

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(CKPT_BASE,  exist_ok=True)
print('Drive mounted.')

In [ ]:
# Clone repo (skip if already present)
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/ibrahim-2337/aerial-segmentation.git {REPO_DIR}

%cd {REPO_DIR}

!pip install -q -r requirements.txt

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup complete.')

## 2. Dataset Download

The INRIA Aerial Image Labeling dataset contains ~280 GeoTIFF files for the training cities we need.  We download via the Kaggle API directly inside Colab to avoid any manual upload steps.

**How it works:**
- The full Kaggle archive is downloaded to Colab's local disk (`/content/`) which has ~200 GB — no Drive quota used during download
- We then copy **only the `train/` directory** (the 5 cities we need) to Google Drive for permanent storage
- The test cities (bellingham, bloomington, etc.) are discarded — we don't need them
- On future sessions, the data is already on Drive and this cell is skipped automatically

**Before running:** Upload your `kaggle.json` API token when prompted.  Get it from kaggle.com → profile → Settings → API → Create New Token.

In [ ]:
import os
from pathlib import Path

DRIVE_DATA = f'{DRIVE_BASE}/data/inria'
TRAIN_IMAGES_ON_DRIVE = f'{DRIVE_DATA}/train/images'

if os.path.isdir(TRAIN_IMAGES_ON_DRIVE) and len(os.listdir(TRAIN_IMAGES_ON_DRIVE)) > 0:
    print(f'Dataset already on Drive ({len(os.listdir(TRAIN_IMAGES_ON_DRIVE))} images). Skipping download.')
else:
    print('Dataset not found on Drive. Starting download...')

    # --- Step 1: Kaggle credentials ---
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print('Upload your kaggle.json when prompted:')
        from google.colab import files
        uploaded = files.upload()   # prompts file picker
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        import shutil
        shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
        print('Kaggle credentials saved.')
    else:
        print('Kaggle credentials already present.')

    # --- Step 2: Install kaggle and download to Colab local disk ---
    !pip install -q kaggle

    LOCAL_TMP = '/content/inria_tmp'
    os.makedirs(LOCAL_TMP, exist_ok=True)

    print('\nDownloading dataset to Colab local disk (this takes ~10-15 min)...')
    !kaggle datasets download \
        -d sagar100rathod/inria-aerial-image-labeling-dataset \
        -p {LOCAL_TMP} --unzip --quiet

    # --- Step 3: Check what we downloaded ---
    src_train = f'{LOCAL_TMP}/AerialImageDataset/train'
    print('\nDownload complete. Sizes:')
    !du -sh {src_train}/images {src_train}/gt

    # --- Step 4: Copy only train/ to Drive (skip test cities) ---
    print('\nCopying train/ to Drive (permanent storage)...')
    os.makedirs(DRIVE_DATA, exist_ok=True)
    !cp -r {src_train} {DRIVE_DATA}/

    # --- Step 5: Free up Colab local disk ---
    import shutil
    shutil.rmtree(LOCAL_TMP)
    print('Cleaned up local tmp.')

    print(f'\nDone. Training images on Drive: {len(os.listdir(TRAIN_IMAGES_ON_DRIVE))}')

# Point DATA_ROOT at Drive — permanent across sessions
DATA_ROOT = DRIVE_DATA
print(f'DATA_ROOT = {DATA_ROOT}')

## 3. Dataset Exploration

The INRIA dataset contains georeferenced GeoTIFF images at 30 cm/pixel resolution, each 5000 × 5000 px.  Ground-truth masks are binary: white (255) = building, black (0) = background.

Here we open one example with `rasterio` to verify the coordinate reference system (CRS) metadata is preserved, then display a tile sample to sanity-check the data pipeline.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

images_dir = Path(DATA_ROOT) / 'train' / 'images'
gt_dir     = Path(DATA_ROOT) / 'train' / 'gt'

tif_files = sorted(images_dir.glob('*.tif'))
print(f'Total training images: {len(tif_files)}')
for f in tif_files[:5]:
    print(f'  {f.name}')

In [ ]:
# Inspect coordinate reference system metadata
sample_img  = tif_files[0]
sample_mask = gt_dir / sample_img.name

with rasterio.open(sample_img) as src:
    print(f'Image : {sample_img.name}')
    print(f'  Size      : {src.width} x {src.height}')
    print(f'  Bands     : {src.count}')
    print(f'  CRS       : {src.crs}')
    print(f'  Transform : {src.transform}')

with rasterio.open(sample_mask) as src:
    mask_data = src.read(1)
    print(f'\nMask : {sample_mask.name}')
    print(f'  Unique values    : {np.unique(mask_data)}')
    print(f'  Building coverage: {(mask_data > 127).mean()*100:.1f}%')

In [ ]:
# Display a 256x256 tile and its ground-truth mask
import rasterio.windows as rw

row, col = 1000, 1500
window   = rw.Window(col, row, 256, 256)

with rasterio.open(sample_img) as src:
    tile_img = src.read(window=window)
with rasterio.open(sample_mask) as src:
    tile_msk = src.read(1, window=window)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.moveaxis(tile_img, 0, -1))
axes[0].set_title('RGB Tile')
axes[1].imshow(tile_msk, cmap='gray')
axes[1].set_title('Ground Truth Mask')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'{sample_img.name}  —  tile @ ({row}, {col})', y=1.02)
plt.tight_layout()
plt.show()
print(f'Building pixels in tile: {(tile_msk > 127).mean()*100:.1f}%')

In [ ]:
# Instantiate the dataset and check tile counts
from src.dataset import InriaDataset, TRAIN_CITIES, VAL_CITIES

train_ds = InriaDataset(DATA_ROOT, split='train', tile_size=256, overlap=0.5)
val_ds   = InriaDataset(DATA_ROOT, split='val',   tile_size=256, overlap=0.5)

print(f'Train cities : {TRAIN_CITIES}')
print(f'Val   cities : {VAL_CITIES}')
print(f'Train tiles  : {len(train_ds):,}')
print(f'Val   tiles  : {len(val_ds):,}')

img_t, msk_t = train_ds[0]
print(f'\nImage tensor : {img_t.shape}  dtype={img_t.dtype}')
print(f'Mask  tensor : {msk_t.shape}  dtype={msk_t.dtype}')

## 4. Training

All three models are trained with `src/train.py`.  We train 2 seeds per model (42 and 0) for 20 epochs each — 6 runs total.

Key design choices:
- **Loss**: BCE + Dice (50/50) handles class imbalance and directly optimises the overlap metric
- **Optimiser**: AdamW with weight decay regularises both CNN and transformer encoders
- **Scheduler**: Cosine annealing decays the learning rate smoothly to near-zero
- **AMP**: Mixed-precision fp16 halves GPU memory and speeds up training on A100
- **Checkpointing**: `last.pth` is overwritten every epoch (resume-safe); `best.pth` updates only on val IoU improvement

Each run takes roughly 45–90 min on an A100.

In [ ]:
import subprocess

runs = [
    ('unet',          42),
    ('unet',           0),
    ('deeplabv3plus', 42),
    ('deeplabv3plus',  0),
    ('segformer',     42),
    ('segformer',      0),
]

for model, seed in runs:
    print(f'\n{"="*60}')
    print(f'Training  model={model}  seed={seed}')
    print(f'{"="*60}')
    result = subprocess.run(
        ['python', 'src/train.py',
         '--model',     model,
         '--seed',      str(seed),
         '--epochs',    '20',
         '--data_root', DATA_ROOT,
         '--resume'],
        capture_output=False
    )
    if result.returncode != 0:
        print(f'[ERROR] training failed for {model}/seed{seed}')

print('\nAll training runs complete.')

## 5. Results — Comparison Table

After training we evaluate every checkpoint on the held-out West Tyrol validation set.  The evaluation script loads the best checkpoint for each model/seed, computes IoU and Dice over all validation tiles, and writes `experiments/results.csv`.

The table aggregates across the two seeds per model showing **mean ± std** to quantify run-to-run variance.

In [ ]:
!python src/evaluate.py --mode metrics

In [ ]:
import pandas as pd
import numpy as np

results_path = f'{REPO_DIR}/experiments/results.csv'
df = pd.read_csv(results_path)

agg = (
    df.groupby('model')[['iou', 'dice']]
      .agg(['mean', 'std'])
      .round(4)
)
agg.columns = ['iou_mean', 'iou_std', 'dice_mean', 'dice_std']
agg = agg.reset_index()
agg['IoU (mean ± std)']  = agg.apply(lambda r: f"{r.iou_mean:.4f} ± {r.iou_std:.4f}",  axis=1)
agg['Dice (mean ± std)'] = agg.apply(lambda r: f"{r.dice_mean:.4f} ± {r.dice_std:.4f}", axis=1)

display_df = agg[['model', 'IoU (mean ± std)', 'Dice (mean ± std)']].rename(columns={'model': 'Architecture'})
print('=== Architecture Comparison — West Tyrol (held-out) ===')
print(display_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, label in zip(axes, ['iou', 'dice'], ['IoU', 'Dice']):
    bars = ax.bar(
        agg['model'], agg[f'{metric}_mean'],
        yerr=agg[f'{metric}_std'],
        color=colors, capsize=6, width=0.5, edgecolor='black', linewidth=0.8,
    )
    ax.set_title(f'Validation {label}', fontsize=13)
    ax.set_ylabel(label)
    ax.set_ylim(0, 1)
    ax.axhline(y=0.70, color='red', linestyle='--', linewidth=1.2, label='70% target')
    ax.legend()
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

fig.suptitle('Aerial Building Segmentation — West Tyrol Validation', fontsize=14)
plt.tight_layout()
os.makedirs(f'{REPO_DIR}/experiments/figures', exist_ok=True)
plt.savefig(f'{REPO_DIR}/experiments/figures/comparison_bar.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Qualitative Visualisation

Metrics alone can obscure failure modes.  Here we load the best checkpoint for each architecture and display side-by-side panels:
- **Input image** — normalised RGB aerial patch
- **Ground truth** — binary building mask from INRIA annotations
- **Prediction** — thresholded sigmoid output (threshold = 0.5)

Per-tile IoU and Dice are shown below each prediction.  Five examples per model.

In [ ]:
!python src/evaluate.py --mode visualize --seed 42
print('Figures saved to experiments/figures/')

In [ ]:
from IPython.display import Image, display

fig_dir = f'{REPO_DIR}/experiments/figures'

for model_name in ['unet', 'deeplabv3plus', 'segformer']:
    fig_path = f'{fig_dir}/{model_name}_seed42_predictions.png'
    if os.path.exists(fig_path):
        print(f'\n--- {model_name.upper()} ---')
        display(Image(filename=fig_path))
    else:
        print(f'[missing] {fig_path}')

## 7. Learning Curves

The training script logs per-epoch metrics (loss, IoU, Dice, lr) to a CSV inside each checkpoint directory.  These curves show convergence speed, overfitting, and optimizer stability across seeds.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

MODELS = ['unet', 'deeplabv3plus', 'segformer']
SEEDS  = [42, 0]

fig, axes = plt.subplots(len(MODELS), 2, figsize=(14, 4 * len(MODELS)))

for row_idx, model in enumerate(MODELS):
    ax_iou, ax_loss = axes[row_idx, 0], axes[row_idx, 1]

    for seed in SEEDS:
        csv_path = f'{CKPT_BASE}/{model}_seed{seed}/metrics.csv'
        if not os.path.exists(csv_path):
            continue
        m = pd.read_csv(csv_path)
        ax_iou.plot(m['epoch'], m['val_iou'],    label=f'seed {seed} val',   linewidth=1.8)
        ax_iou.plot(m['epoch'], m['train_iou'],  label=f'seed {seed} train', linewidth=1.2, linestyle='--')
        ax_loss.plot(m['epoch'], m['val_loss'],  label=f'seed {seed} val')
        ax_loss.plot(m['epoch'], m['train_loss'],label=f'seed {seed} train', linestyle='--')

    ax_iou.axhline(0.70, color='red', linestyle=':', linewidth=1.2, label='70% target')
    ax_iou.set_title(f'{model} — Validation IoU', fontsize=11)
    ax_iou.set_xlabel('Epoch')
    ax_iou.set_ylabel('IoU')
    ax_iou.legend(fontsize=7)
    ax_iou.set_ylim(0, 1)

    ax_loss.set_title(f'{model} — Loss', fontsize=11)
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('BCE+Dice Loss')
    ax_loss.legend(fontsize=7)

plt.suptitle('Training Curves — All Models', fontsize=14)
plt.tight_layout()
plt.savefig(f'{REPO_DIR}/experiments/figures/learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()